# NC-04: Node Classification with Pretrained GraphSAGE Model

Loads the MSD-prepared CSV dataset, runs inference with the pretrained
`msd_node_classifier.pt` GraphSAGE model, and visualises which room types
the model predicts correctly.

**Run NC-02 first** to generate the CSV files in `graphs/`.

In [25]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

from topologicpy.PyG import PyG
from topologicpy.Helper import Helper
from topologicpy.Graph import Graph
from topologicpy.Dictionary import Dictionary
from topologicpy.Topology import Topology


## 0. Renderer & version

In [26]:
renderer = 'vscode'
print('TopologicPy version:', Helper.Version())

TopologicPy version: The version that you are using (0.9.51) is EQUAL TO the latest version available on PyPI.


## 1. Paths

In [27]:
BASE         = Path(r'E:\softwares-4\graph-ml\assign-04-node-classification')
DATASET_PATH = BASE / 'graphs'
MODEL_PATH   = BASE / 'model' / 'msd_node_classifier.pt'

# Task definition
PREDICTION_LEVEL  = 'node'
TASK              = 'classification'
GRAPH_LABEL_TYPE  = 'categorical'
NODE_LABEL_TYPE   = 'categorical'
EDGE_LABEL_TYPE   = 'categorical'

ROOM_LABEL_MAP = {
    0: 'bedroom',
    1: 'livingroom',
    2: 'kitchen',
    3: 'dining',
    4: 'corridor',
    5: 'stairs',
    6: 'storeroom',
    7: 'bathroom',
    8: 'balcony',
}

print('Dataset :', DATASET_PATH)
print('Model   :', MODEL_PATH)
print('Model exists:', MODEL_PATH.exists())

Dataset : E:\softwares-4\graph-ml\assign-04-node-classification\graphs
Model   : E:\softwares-4\graph-ml\assign-04-node-classification\model\msd_node_classifier.pt
Model exists: True


## 2. Check dataset files

In [28]:
required_files = [
    DATASET_PATH / 'graphs.csv',
    DATASET_PATH / 'nodes.csv',
    DATASET_PATH / 'edges.csv',
]

for f in required_files:
    print(f'{f.name}: {"FOUND" if f.exists() else "MISSING"} -> {f}')

if not all(f.exists() for f in required_files):
    raise FileNotFoundError('One or more CSV files are missing. Run NC-02 first.')

graphs.csv: FOUND -> E:\softwares-4\graph-ml\assign-04-node-classification\graphs\graphs.csv
nodes.csv: FOUND -> E:\softwares-4\graph-ml\assign-04-node-classification\graphs\nodes.csv
edges.csv: FOUND -> E:\softwares-4\graph-ml\assign-04-node-classification\graphs\edges.csv


## 3. Inspect CSV schema

In [29]:
graphs_df = pd.read_csv(DATASET_PATH / 'graphs.csv')
nodes_df  = pd.read_csv(DATASET_PATH / 'nodes.csv')
edges_df  = pd.read_csv(DATASET_PATH / 'edges.csv')

print('graphs.csv shape:', graphs_df.shape)
print('nodes.csv  shape:', nodes_df.shape)
print('edges.csv  shape:', edges_df.shape)
print()
print('nodes.csv columns:', list(nodes_df.columns))
print()
print(nodes_df.to_string(index=False))

graphs.csv shape: (1, 2)
nodes.csv  shape: (10, 16)
edges.csv  shape: (26, 7)

nodes.csv columns: ['graph_id', 'node_id', 'label', 'train_mask', 'val_mask', 'test_mask', 'feat_feat_zoning_type_0', 'feat_feat_zoning_type_1', 'feat_feat_zoning_type_2', 'feat_feat_zoning_type_3', 'feat_feat_connectivity_0', 'feat_feat_connectivity_1', 'feat_feat_connectivity_2', 'X', 'Y', 'Z']

 graph_id  node_id  label  train_mask  val_mask  test_mask  feat_feat_zoning_type_0  feat_feat_zoning_type_1  feat_feat_zoning_type_2  feat_feat_zoning_type_3  feat_feat_connectivity_0  feat_feat_connectivity_1  feat_feat_connectivity_2            X            Y      Z
        0        0      7       False     False       True                      0.0                      0.0                      1.0                      0.0                       0.0                       1.0                       0.0  6410.993164   740.000000 1500.0
        0        1      1       False     False       True                      0.

## 4. Load dataset

In [30]:
pyg = PyG.ByCSVPath(
    path=str(DATASET_PATH),
    level=PREDICTION_LEVEL,
    task=TASK,
    graphLabelType=GRAPH_LABEL_TYPE,
    nodeLabelType=NODE_LABEL_TYPE,
    edgeLabelType=EDGE_LABEL_TYPE,
)
print('Dataset loaded.')

Dataset loaded.


## 5. Load pre-trained model

In [31]:
pyg.LoadModel(str(MODEL_PATH))
print('Model loaded.')

Model loaded.


## 6. Predict node labels

Runs inference over all nodes (no training — `split='all'` treats everything as test).

In [32]:
_ = pyg.Predict(split='all', return_probs=True, attach_to_data=True)
print('Predictions attached to the dataset.')

Predictions attached to the dataset.


## 7. Export node predictions to CSV

In [33]:
def _to_class_index(value):
    arr = np.asarray(value)
    arr = np.squeeze(arr)
    if arr.ndim == 0:
        return int(arr)
    if arr.ndim == 1:
        if arr.size == 1:
            return int(arr[0])
        return int(np.argmax(arr))
    raise ValueError(f'Cannot convert shape {arr.shape} to class index.')


def export_node_predictions(pyg_obj, output_csv):
    pred_report  = pyg_obj.Predict(split='all', return_probs=True, attach_to_data=True)
    pred_by_graph   = pred_report['pred']
    y_true_by_graph = pred_report['y_true']
    prob_by_graph   = pred_report.get('prob', None)
    rows = []
    for graph_idx, data in enumerate(pyg_obj.data_list):
        graph_id   = int(data.graph_id.item()) if hasattr(data, 'graph_id') else graph_idx
        n          = data.num_nodes
        graph_pred = np.asarray(pred_by_graph[graph_idx])
        graph_true = np.asarray(y_true_by_graph[graph_idx])
        graph_prob = np.asarray(prob_by_graph[graph_idx]) if prob_by_graph is not None else None
        train_mask = data.train_mask.detach().cpu().numpy() if hasattr(data, 'train_mask') else None
        val_mask   = data.val_mask.detach().cpu().numpy()   if hasattr(data, 'val_mask')   else None
        test_mask  = data.test_mask.detach().cpu().numpy()  if hasattr(data, 'test_mask')  else None
        for node_idx in range(n):
            y_true_i = _to_class_index(graph_true[node_idx])
            y_pred_i = _to_class_index(graph_pred[node_idx])
            row = {'graph_id': graph_id, 'node_id': node_idx, 'y_true': y_true_i, 'y_pred': y_pred_i}
            if graph_prob is not None:
                prob_i = np.asarray(graph_prob[node_idx]).squeeze()
                if prob_i.ndim == 1 and y_pred_i < prob_i.size:
                    row['y_pred_prob'] = round(float(prob_i[y_pred_i]), 4)
            if train_mask is not None: row['train_mask'] = bool(train_mask[node_idx])
            if val_mask   is not None: row['val_mask']   = bool(val_mask[node_idx])
            if test_mask  is not None: row['test_mask']  = bool(test_mask[node_idx])
            rows.append(row)
    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)
    return df


predictions_csv = DATASET_PATH / 'node_predictions.csv'
predictions_df  = export_node_predictions(pyg, predictions_csv)
print(f'Saved predictions to: {predictions_csv}')

Saved predictions to: E:\softwares-4\graph-ml\assign-04-node-classification\graphs\node_predictions.csv


## 8. Results table

In [34]:
# Map integer labels to room-type names
predictions_df['true_label'] = predictions_df['y_true'].map(ROOM_LABEL_MAP)
predictions_df['pred_label'] = predictions_df['y_pred'].map(ROOM_LABEL_MAP)
predictions_df['correct']    = predictions_df['y_true'] == predictions_df['y_pred']

print(predictions_df[['node_id', 'true_label', 'pred_label', 'y_pred_prob', 'correct']].to_string(index=False))

 node_id true_label pred_label  y_pred_prob  correct
       0   bathroom    bedroom       0.9975    False
       1 livingroom    bedroom       0.9975    False
       2    bedroom    bedroom       0.9975     True
       3     dining    bedroom       0.9975    False
       4   bathroom    bedroom       0.9975    False
       5    kitchen    bedroom       0.9975    False
       6   corridor    bedroom       0.9975    False
       7    bedroom    bedroom       0.9975     True
       8    bedroom    bedroom       0.9975     True
       9   corridor    bedroom       0.9975    False


## 9. Accuracy summary

In [35]:
total   = len(predictions_df)
correct = predictions_df['correct'].sum()
accuracy = correct / total if total > 0 else 0.0

print(f'Nodes classified : {total}')
print(f'Correct          : {correct}')
print(f'Accuracy         : {accuracy:.1%}')

print()
print('Per-class breakdown:')
breakdown = predictions_df.groupby('true_label').agg(
    total=('correct', 'count'),
    correct=('correct', 'sum')
).reset_index()
breakdown['accuracy'] = (breakdown['correct'] / breakdown['total']).round(3)
print(breakdown.to_string(index=False))

Nodes classified : 10
Correct          : 3
Accuracy         : 30.0%

Per-class breakdown:
true_label  total  correct  accuracy
  bathroom      2        0       0.0
   bedroom      3        3       1.0
  corridor      2        0       0.0
    dining      1        0       0.0
   kitchen      1        0       0.0
livingroom      1        0       0.0


## 10. Visualise predictions on graph

Reloads the graph from CSV and colours vertices by true label vs predicted label.
Misclassified nodes appear larger and red.

In [ ]:
import pandas as pd
from topologicpy.Color import Color

ROOM_COLORS = {
    0: '#4A90D9',   # bedroom
    1: '#E8873A',   # livingroom
    2: '#F5C842',   # kitchen
    3: '#5DBB63',   # dining
    4: '#9B9B9B',   # corridor
    5: '#9B59B6',   # stairs
    6: '#8B6343',   # storeroom
    7: '#3EC9C9',   # bathroom
    8: '#A8E6A3',   # balcony
}

ROOM_ABBR = {
    0: 'BED', 1: 'LIV', 2: 'KIT', 3: 'DIN',
    4: 'COR', 5: 'STR', 6: 'STO', 7: 'BAT', 8: 'BAL',
}

preds_df = pd.read_csv(DATASET_PATH / 'node_predictions.csv')
pred_map = dict(zip(preds_df['node_id'], preds_df['y_pred']))

vis_graphs = Graph.ByCSVPath(path=str(DATASET_PATH))
g = vis_graphs[0]

verts = Graph.Vertices(g) or []
for v in verts:
    d       = Topology.Dictionary(v)
    true    = Dictionary.ValueAtKey(d, 'label')
    node_id = Dictionary.ValueAtKey(d, 'node_id')
    if true is None or node_id is None:
        continue
    true    = int(true)
    node_id = int(node_id)
    pred    = int(pred_map.get(node_id, true))
    is_wrong = true != pred

    true_color      = ROOM_COLORS.get(true, '#888888')
    pred_show_color = 'red' if is_wrong else ROOM_COLORS.get(pred, '#888888')
    border_color    = 'red' if is_wrong else '#333333'
    border_width    = 6 if is_wrong else 1
    true_name       = ROOM_ABBR.get(true, str(true))
    pred_name       = ROOM_ABBR.get(pred, str(pred))
    overlay_label   = f'{true_name}→{pred_name}' if is_wrong else true_name

    d = Dictionary.SetValuesAtKeys(d,
        ['true_color', 'pred_show_color', 'border_color', 'border_width',
         'true', 'pred', 'true_name', 'pred_name', 'overlay_label'],
        [true_color, pred_show_color, border_color, border_width,
         true, pred, true_name, pred_name, overlay_label])
    v = Topology.SetDictionary(v, d)

g = Graph.Reshape(g)

print('--- True Labels ---')
Topology.Show(g, vertexSize=25, vertexColorKey='true_color',
              showVertexLabel=True, vertexLabelKey='true_name',
              backgroundColor='white', camera=[0, 0, 3],
              vertexLabelFontSize=14, renderer=renderer)

print('--- Predicted Labels (red = misclassified) ---')
Topology.Show(g, vertexSize=25, vertexColorKey='pred_show_color',
              showVertexLabel=True, vertexLabelKey='pred_name',
              backgroundColor='white', camera=[0, 0, 3],
              vertexLabelFontSize=14, renderer=renderer)

print('--- Overlay: true room color, red border = misclassified, label = true→pred ---')
Topology.Show(g, vertexSize=25, vertexColorKey='true_color',
              vertexBorderColorKey='border_color', vertexBorderWidthKey='border_width',
              showVertexLabel=True, vertexLabelKey='overlay_label',
              backgroundColor='white', camera=[0, 0, 3],
              vertexLabelFontSize=14, renderer=renderer)